In [15]:
import os
import json
import requests
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader

In [16]:
load_dotenv(override=True)
openai = OpenAI()

Using Pushover for notification

In [17]:
pushover_user_token = os.getenv("PUSHOVER_USER_KEY")
pushover_api_key = os.getenv("PUSH_OVER_API_KEY")
pushover_url = "https://api.pushover.net/1/messages.json"

In [18]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user_token, "token": pushover_api_key, "message": message}
    requests.post(pushover_url, data=payload)

In [9]:
push("Hello...")

Push: Hello...


In [19]:
def record_user_detail(email, name="unknown name", message="unknown message"):
    push(f"Recording detail from {name} with email {email} has message : {message}")
    return {"recorded":"ok"}

In [20]:
def record_user_question(question):
    push(f"Recording {question} that was asked to me and I am not able to answer it.")
    return {"recorded": "ok"}   

Now lets create user_detail and user_question json blobs for tooling

In [21]:
record_user_detail_json = {
    "name":"record_user_detail",
    "description":"Provides user details with name, email and notes",
    "parameters": {
        "type": "object",
        "properties":{
            "email": {
                "type": "string",
                "description": "email of the user"
            },
            "name":{
                "type": "string",
                "description": "name of the user"
            },
            "message":{
                "type": "string",
                "description": "message provided by user"
            }
        },
        "required": ["email"],
        "additionalProperties": False                
    }
}

In [23]:
record_user_question_json = {
    "name": "record_user_question",
    "description": "Records a question or notes from user",
    "parameters": {
        "type": "object",
        "properties":{
            "question":{
            "type": "string",
            "description": "Question or notes from a user"
        },                    
        },        
    },
    "required":["question"],
    "additionalProperties": False
}

In [24]:
tools = [
    {"type": "function", "function": record_user_question_json},
    {"type": "function", "function": record_user_detail_json}
]

In [41]:
globals()["record_user_question"]("How you will be 5 years down the lane..")

Push: Recording How you will be 5 years down the lane.. that was asked to me and I am not able to answer it.


{'recorded': 'ok'}

In [25]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name        
        arguments = json.loads(tool_call.function.arguments) 
        print(f"Tool Called -> {tool_name}")
        if tool_name == "record_user_question":
            result = record_user_question(**arguments)
        elif tool_name == "record_user_detail":
            result = record_user_detail(**arguments)                          
        
        results.append({"role":"tool", "content": json.dumps(result), "tool_call_id": tool_call.id})
    return results

In [32]:
reader = PdfReader("me/Somashekhar_Muniyappa_TX.pdf")
my_profile = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        my_profile += text
with open("me/Somashekhar_Muniyappa_TX_summary.txt", "r", encoding="utf-8") as s:
    summary = s.read()
name = "Soma"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and resume which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer to any question, use your record_user_question tool to record the question that you couldn't answer, even if it's about something trivial or unrelated to career. \
Respond to user saying {name} is alerted about this question and stop conversation by adding stop to finish_reason\
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email, name and a message to record it using your record_user_detail tool. \
Respond to user saying {name} is alerted with your email and stop conversation by adding stop to finish_reason"

system_prompt += f"\n\n## Summary:\n{summary}\n\n## Resume:\n{my_profile}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [30]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    done = False
    max_iteration = 1
    iteration = 0
    while not done and iteration <= max_iteration:        
        response = openai.chat.completions.create(model= "gpt-4o-mini", messages= messages, tools = tools)
        finish_reason = response.choices[0].finish_reason 
        print(f"Finish Reason -> {finish_reason}")      
        iteration += 1
        if finish_reason=="tool_calls":            
            message = response.choices[0].message            
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results) 
        else:
            done = True   
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Finish Reason -> tool_calls
Tool Called -> record_user_question
Push: Recording What is Soma's favorite color? that was asked to me and I am not able to answer it.
Finish Reason -> stop
Finish Reason -> stop
Finish Reason -> tool_calls
Tool Called -> record_user_detail
Push: Recording detail from Shekhar with email ss@gmail.com has message : I want to work with you
Finish Reason -> stop
Finish Reason -> tool_calls
Tool Called -> record_user_question
Push: Recording can you figure out how to fix the efi boot when it breaks or does not work? that was asked to me and I am not able to answer it.
Finish Reason -> stop
Finish Reason -> stop
Finish Reason -> stop
Finish Reason -> stop
Finish Reason -> tool_calls
Tool Called -> record_user_question
Push: Recording User mentioned 'pwa=true'. that was asked to me and I am not able to answer it.
Finish Reason -> stop
Finish Reason -> tool_calls
Tool Called -> record_user_question
Push: Recording How to fix EFI issues when downloading Windows 11 o